In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_DIM_CASES_INFO_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_CASES AS
    SELECT DISTINCT
        c.CASE_TYPE AS CASE_TYPE_AV_ID,
        av1.VALUE_SHORT_DESC AS CASE_TYPE_DESC,
        c.CURRENT_CASE_STATUS_CODE AS CURRENT_CASE_STATUS_CODE_AV_ID,
        av2.VALUE_SHORT_DESC AS CURRENT_CASE_STATUS_CODE_DESC,
        c.RESTRICTION_REASON_CODE AS RESTRICTION_REASON_CODE_AV_ID,
        av3.VALUE_SHORT_DESC AS RESTRICTION_REASON_CODE_DESC,
        c.DELETED_DATE,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(c.CASE_TYPE), ''),
            COALESCE(av1.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.CURRENT_CASE_STATUS_CODE), ''),
            COALESCE(av2.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.RESTRICTION_REASON_CODE), ''),
            COALESCE(av3.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.DELETED_DATE), '')
        )) AS CHECKSUM
    FROM {STG}.{STG_CASES} c
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av1
        ON c.CASE_TYPE = av1.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av2
        ON c.CURRENT_CASE_STATUS_CODE = av2.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av3
        ON c.RESTRICTION_REASON_CODE = av3.AV_ID
    """).collect()

    session.sql(f"""
    UPDATE {DWH}.{DIM_CASES_INFO} tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_CASES src
    WHERE tgt.CASE_TYPE_AV_ID IS NOT DISTINCT FROM src.CASE_TYPE_AV_ID
      AND tgt.CURRENT_CASE_STATUS_CODE_AV_ID IS NOT DISTINCT FROM src.CURRENT_CASE_STATUS_CODE_AV_ID
      AND tgt.RESTRICTION_REASON_CODE_AV_ID IS NOT DISTINCT FROM src.RESTRICTION_REASON_CODE_AV_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    insert_result = session.sql(f"""
    INSERT INTO {DWH}.{DIM_CASES_INFO}
    (
        CASE_TYPE_AV_ID,
        CASE_TYPE_DESC,
        CURRENT_CASE_STATUS_CODE_AV_ID,
        CURRENT_CASE_STATUS_CODE_DESC,
        RESTRICTION_REASON_CODE_AV_ID,
        RESTRICTION_REASON_CODE_DESC,
        CREATED_DATE,
        UPDATED_DATE,
        DELETED_DATE,
        CHECKSUM
    )
    SELECT
        src.CASE_TYPE_AV_ID,
        src.CASE_TYPE_DESC,
        src.CURRENT_CASE_STATUS_CODE_AV_ID,
        src.CURRENT_CASE_STATUS_CODE_DESC,
        src.RESTRICTION_REASON_CODE_AV_ID,
        src.RESTRICTION_REASON_CODE_DESC,
        CURRENT_TIMESTAMP(),
        NULL,
        src.DELETED_DATE,
        src.CHECKSUM
    FROM TEMP_CASES src
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM {DWH}.{DIM_CASES_INFO} tgt
        WHERE tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_CASES_INFO load completed. Rows Loaded: {rows_loaded}"
    )

    print(f"DWH LOAD SUCCESS | Rows Loaded: {rows_loaded}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_CASES_INFO load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")